# GOT OCR single-page smoke test

This notebook runs a minimal, reproducible OCR test for exactly one PDF page.
It follows the same project imports used by the pipeline and adds conservative limits so generation does not run for too long.

In [1]:
import os
import sys
import gc
import time
import tempfile
import types
from pathlib import Path

import torch
from pdf2image import convert_from_path
from transformers import AutoModel, StoppingCriteria, TextStreamer
from transformers.cache_utils import DynamicCache

# Ensure project root is importable from notebooks/.
ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Match notebook behavior from prior attempts.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Compatibility shim for GOT remote code in some transformers versions.
if not hasattr(DynamicCache, "seen_tokens"):
    DynamicCache.seen_tokens = property(
        lambda self: getattr(self, "_seen_tokens", 0),
        lambda self, value: setattr(self, "_seen_tokens", value),
    )

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [2]:
from src.utils.got_model import get_got_model

MODEL_ID = "stepfun-ai/GOT-OCR2_0"

# Cleanup from prior runs in this kernel.
if "GOT_MODEL" in globals():
    del GOT_MODEL
if "GOT_TOKENIZER" in globals():
    del GOT_TOKENIZER
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if not torch.cuda.is_available():
    raise RuntimeError("CUDA nao disponivel. Este notebook esta configurado para rodar apenas na GPU.")

# GPU-only load (same path used by pipeline util).
GOT_MODEL, GOT_TOKENIZER = get_got_model(None, None)
DEVICE = "cuda"

print("model device:", DEVICE)


[INFO] Carregando GOT-OCR 2.0 (primeira execucao faz download ~1.5GB)...


c:\Users\bella\miniconda3\envs\study\Lib\site-packages\transformers\modeling_utils.py:5198: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  _ = torch.empty(int(byte_count // 2), dtype=torch.float16, device=device, requires_grad=False)


Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

model device: cuda


In [5]:
PDF_PATH = ROOT / "data" / "topik_papers" / "102nd-topik-test-papers" / "102nd-TOPIK-II-Reading-Test-Paper.pdf"
PAGE_NUMBER = 4
DPI = 150
MAX_IMAGE_SIDE = 1024
MAX_NEW_TOKENS = 512
NO_REPEAT_NGRAM_SIZE = 8
MAX_SECONDS = 120  # hard wall-clock cap; this GPU is power-limited and can be slow per token

if DEVICE != "cuda":
    raise RuntimeError("Execucao bloqueada: este teste aceita apenas GPU (CUDA).")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

page_image = convert_from_path(
    str(PDF_PATH),
    first_page=PAGE_NUMBER,
    last_page=PAGE_NUMBER,
    dpi=DPI,
    grayscale=True,
 )[0]
page_image.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE))
print("page size:", page_image.size)

# Classes/constants for prompt + image preprocessing live in the GOT remote code module.
_got_module = sys.modules[GOT_MODEL.__class__.__module__]
Conversation = _got_module.Conversation
SeparatorStyle = _got_module.SeparatorStyle
KeywordsStoppingCriteria = _got_module.KeywordsStoppingCriteria
GOTImageEvalProcessor = _got_module.GOTImageEvalProcessor
DEFAULT_IM_START_TOKEN = _got_module.DEFAULT_IM_START_TOKEN
DEFAULT_IMAGE_PATCH_TOKEN = _got_module.DEFAULT_IMAGE_PATCH_TOKEN
DEFAULT_IM_END_TOKEN = _got_module.DEFAULT_IM_END_TOKEN

image_token_len = 256
qs = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_PATCH_TOKEN * image_token_len + DEFAULT_IM_END_TOKEN + "\nOCR: "

conv = Conversation(
    system="""<|im_start|>system
    You should follow the instructions carefully and explain your answers in detail.""",
    roles=("<|im_start|>user\n", "<|im_start|>assistant\n"),
    version="mpt",
    messages=[],
    offset=0,
    sep_style=SeparatorStyle.MPT,
    sep="<|im_end|>",
)
conv.append_message(conv.roles[0], qs)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()

input_ids = torch.as_tensor(GOT_TOKENIZER([prompt]).input_ids).cuda()
# GOTImageEvalProcessor normalizes with 3-channel mean/std; convert the grayscale page to RGB.
image_tensor = GOTImageEvalProcessor(image_size=1024)(page_image.convert("RGB")).unsqueeze(0).half().cuda()

stop_str = conv.sep
stopping_criteria = KeywordsStoppingCriteria([stop_str], GOT_TOKENIZER, input_ids)


class MaxTimeCriteria(StoppingCriteria):
    """Wall-clock cap independent of token count, since per-token speed varies with GPU load."""

    def __init__(self, max_seconds):
        self.deadline = time.perf_counter() + max_seconds

    def __call__(self, input_ids, scores, **kwargs):
        return time.perf_counter() > self.deadline


time_criteria = MaxTimeCriteria(MAX_SECONDS)
# Streams tokens to stdout as they're generated, proof that it's progressing and not stuck.
streamer = TextStreamer(GOT_TOKENIZER, skip_prompt=True, skip_special_tokens=True)

t0 = time.perf_counter()
try:
    torch.cuda.empty_cache()
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        # Direct generate() call so MAX_NEW_TOKENS hard-caps output length (chat() hardcodes 4096).
        output_ids = GOT_MODEL.generate(
            input_ids,
            images=[image_tensor],
            do_sample=False,
            num_beams=1,
            no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
            max_new_tokens=MAX_NEW_TOKENS,
            stopping_criteria=[stopping_criteria, time_criteria],
            streamer=streamer,
        )
except torch.cuda.OutOfMemoryError as exc:
    raise RuntimeError(
        "CUDA OOM no modo GPU-only. Reduza DPI/MAX_IMAGE_SIDE ou MAX_NEW_TOKENS para continuar na GPU."
    ) from exc
elapsed = time.perf_counter() - t0

text = GOT_TOKENIZER.decode(output_ids[0, input_ids.shape[1]:]).strip()
if text.endswith(stop_str):
    text = text[: -len(stop_str)].strip()

print(f"device used: {DEVICE}")
print(f"elapsed: {elapsed:.1f}s")
print(f"generated tokens: {output_ids.shape[1] - input_ids.shape[1]}")
print("-" * 80)
print(text[:4000])
if len(text) > 4000:
    print("\n[output truncated at 4000 chars]")


page size: (724, 1024)
2
2
2
2
        2
2
2
22
2
2
2 
2
2
2
20
2
device used: cuda
elapsed: 121.8s
generated tokens: 41
--------------------------------------------------------------------------------
2
2
2
2
        2
2
2
22
2
2
2 
2
2
2
20
2


In [6]:
import numpy as np

debug_path = ROOT / "outputs" / "figures" / f"debug_page_{PAGE_NUMBER}.png"
debug_path.parent.mkdir(parents=True, exist_ok=True)
page_image.save(debug_path)
arr = np.array(page_image.convert("L"))
print("saved to:", debug_path)
print("mode:", page_image.mode, "size:", page_image.size)
print("pixel mean/min/max:", arr.mean(), arr.min(), arr.max())
print("fraction near-white (>240):", (arr > 240).mean())


saved to: C:\Users\bella\Desktop\Pessoal\Estudo\korean-study\outputs\figures\debug_page_4.png
mode: L size: (724, 1024)
pixel mean/min/max: 244.41555701830112 0 255
fraction near-white (>240): 0.8172978353763812
